# SDRB 2026-1 deal structuring sandbox

This notebook runs the synthetic DSRB Sovereign Defence and Resilience Bond engine. It is meant for structuring work: load the config, simulate the asset pool, run the waterfall, price the senior guarantee, optimise the stack, then plot base vs severe stress.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_engine import SovereignAssetPool, load_deal_config
from src.structural_waterfall import StructuredWaterfallEngine
from src.credit_enhancement import DSRBGuaranteePricer, guarantee_summary_frame
from src.optimization_engine import CapitalStructureOptimizer


## 1. Load the deal configuration

The YAML file controls the issuance amount, collateral target, tranche thicknesses, coupons, macro shock settings, guarantee terms, and optimiser bounds.


In [ ]:
CONFIG_PATH = ROOT / 'config' / 'deal_structure.yaml'
config = load_deal_config(CONFIG_PATH)
config['deal']


## 2. Generate the synthetic sovereign asset pool

The collateral tape has 50+ sovereign-backed defence and dual-use infrastructure exposures. It is synthetic, but the fields are the ones needed for tranche modelling: principal, coupon, maturity, PD, LGD, sector, country, and macro sensitivity.


In [ ]:
pool = SovereignAssetPool.from_config(CONFIG_PATH)
assets = pool.asset_pool
pool.pool_summary()


In [ ]:
assets.head(10)


## 3. Run base and severe stress simulations

For faster notebook work, start with 1,000 paths. For a full run, switch `N_PATHS` to 10,000.


In [ ]:
N_PATHS = 1000
CHUNK_SIZE = 500

base_sim = pool.simulate_portfolio(
    n_paths=N_PATHS,
    macro_config=config['macro'],
    scenario='base',
    chunk_size=CHUNK_SIZE,
)

severe_sim = pool.simulate_portfolio(
    n_paths=N_PATHS,
    macro_config=config['macro'],
    scenario='severe',
    chunk_size=CHUNK_SIZE,
)

base_sim.net_loss_rate.mean(), severe_sim.net_loss_rate.mean()


## 4. Compare portfolio cashflows under base vs severe stress


In [ ]:
months = np.arange(1, config['deal']['horizon_months'] + 1)

plt.figure(figsize=(10, 5))
plt.plot(months, base_sim.total_collections.mean(axis=0) / 1e6, label='base collections')
plt.plot(months, severe_sim.total_collections.mean(axis=0) / 1e6, label='severe collections')
plt.xlabel('Month')
plt.ylabel('Average monthly collections, USD mm')
plt.title('Portfolio cashflows: base vs severe stress')
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(months, base_sim.collateral_balance.mean(axis=0) / 1e9, label='base collateral balance')
plt.plot(months, severe_sim.collateral_balance.mean(axis=0) / 1e9, label='severe collateral balance')
plt.xlabel('Month')
plt.ylabel('Average collateral balance, USD bn')
plt.title('Collateral roll-down')
plt.legend()
plt.show()


## 5. Run the structured waterfall

The engine pays fees, interest, principal, PDL cures, reserve top-ups, trapped cash, and equity excess spread in order.


In [ ]:
waterfall = StructuredWaterfallEngine.from_config(CONFIG_PATH)
base_wf = waterfall.run(base_sim)
severe_wf = waterfall.run(severe_sim)

base_wf.tranche_loss_rates().describe(percentiles=[0.5, 0.95, 0.99])


In [ ]:
severe_wf.tranche_loss_rates().describe(percentiles=[0.5, 0.95, 0.99])


## 6. Plot tranche loss distributions


In [ ]:
losses = severe_wf.tranche_loss_rates()

plt.figure(figsize=(10, 5))
plt.hist(losses['equity_loss_rate'], bins=40, alpha=0.75, label='equity')
plt.hist(losses['mezzanine_loss_rate'], bins=40, alpha=0.75, label='mezzanine')
plt.hist(losses['senior_loss_rate'], bins=40, alpha=0.75, label='senior')
plt.xlabel('Terminal loss rate')
plt.ylabel('Path count')
plt.title('Tranche loss distribution under severe stress')
plt.legend()
plt.show()


## 7. Price the DSRB senior guarantee

The guarantee fee is based on expected discounted senior loss, capped by the agreed guarantee notional, with capital and admin loading added in bps.


In [ ]:
pricer = DSRBGuaranteePricer.from_config(CONFIG_PATH)
base_guarantee = pricer.price_from_waterfall(base_wf)
severe_guarantee = pricer.price_from_waterfall(severe_wf)

pd.concat(
    [
        guarantee_summary_frame(base_guarantee).assign(scenario='base'),
        guarantee_summary_frame(severe_guarantee).assign(scenario='severe'),
    ],
    ignore_index=True,
)


## 8. Optimise the capital stack

This solves for senior thickness, mezzanine thickness, and coupons that minimise issuer WACC while satisfying yield hurdles and senior subordination.


In [ ]:
optimizer = CapitalStructureOptimizer.from_config(CONFIG_PATH)
opt = optimizer.optimise(base_sim)

{
    'success': opt.success,
    'message': opt.message,
    'senior_thickness': opt.senior_thickness,
    'mezzanine_thickness': opt.mezzanine_thickness,
    'equity_thickness': opt.equity_thickness,
    'senior_coupon': opt.senior_coupon,
    'mezzanine_coupon': opt.mezzanine_coupon,
    'wacc': opt.optimized_wacc,
}


In [ ]:
opt.tranche_metrics


## 9. Inspect trigger breaches

A clean structure should not just have a nice base case. The trigger table shows when OC and trapping rules actually bite.


In [ ]:
severe_wf.trigger_breaches.head(20)
